# LLaMA 3.1 8B. Evaluation Only (cross-family validation)

**Use this notebook if you already have `results/corpus_llama_d2.jsonl` checked into the repo** (197 samples). It skips corpus generation and runs only the evaluations needed to fill in the cross-model numbers in the report.

Total runtime on a T4: ~45 min.

**Prerequisites:**
1. Runtime → Change runtime type → **T4 GPU** (free tier OK).
2. HuggingFace token saved in Colab Secrets as `HF_TOKEN`. Make sure you've accepted the LLaMA license at <https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct>.
3. Run cells top-to-bottom. Each step downloads results to your local machine at the end.

After it finishes, drop the downloaded zip into `results/` on `main` and the report figures will regenerate automatically.

## Cell 1. Setup (clone repo, install deps, login)

In [ ]:
import os, sys

GITHUB_URL = 'https://github.com/AliHasan-786/llm-invisible-watermarking.git'
REPO_DIR   = '/content/llm-invisible-watermarking'
MODEL      = 'meta-llama/Llama-3.1-8B-Instruct'

if not os.path.exists(REPO_DIR):
    print('Cloning main...')
    os.system(f'git clone {GITHUB_URL} {REPO_DIR}')
else:
    print('Pulling latest...')
    os.system(f'cd {REPO_DIR} && git pull')

os.chdir(REPO_DIR)
sys.path.insert(0, '.')
print('Working dir:', os.getcwd())

!pip install -q 'transformers>=4.45' datasets accelerate sentencepiece protobuf scipy matplotlib

from google.colab import userdata
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
os.environ['HUGGING_FACE_HUB_TOKEN'] = os.environ['HF_TOKEN']
from huggingface_hub import login
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('HF login OK')

!nvidia-smi | head -12
!ls -la results/

## Cell 2. Verify the LLaMA corpus is present (no generation needed)

Expect ~197 samples (99 watermarked + 98 unwatermarked). If you see this, you can skip corpus generation entirely and go straight to evaluation.

In [ ]:
from pipeline.generate import load_corpus
import numpy as np

corpus = load_corpus('results/corpus_llama_d2.jsonl')
wm  = [x for x in corpus if x['watermarked']]
uwm = [x for x in corpus if not x['watermarked']]
print(f'Watermarked:   {len(wm)}')
print(f'Unwatermarked: {len(uwm)}')
print(f'Avg tokens. wm: {np.mean([x["n_tokens"] for x in wm]):.0f}, uwm: {np.mean([x["n_tokens"] for x in uwm]):.0f}')
print(f'Sources: {set(x["source"] for x in corpus)}')
assert len(wm) > 50 and len(uwm) > 50, 'Corpus too small. run scripts/generate_corpus.py first'
print('OK. corpus ready for evaluation')

## Cell 3. Detection + perplexity eval (~20 min)

Calibrates a z-score threshold at 1% FPR on the unwatermarked half, reports TPR all-lengths and TPR for ≥150 tokens, scores GPT-2 perplexity on watermarked vs unwatermarked.

Outputs:
- `results/detection_llama_summary.json`
- `results/detection_llama_zscores.npz`

In [ ]:
!python scripts/eval_detection.py --model meta-llama/Llama-3.1-8B-Instruct

In [ ]:
import json
with open('results/detection_llama_summary.json') as f:
    s = json.load(f)
print(f"Calibrated z-threshold:  {s['calibrated_z_threshold']:.3f}")
print(f"TPR (all lengths):       {s['tpr_at_1pct_fpr_all']:.1%}")
long_key = next((k for k in s if 'ge150' in k), None)
if long_key:
    print(f"TPR (>=150 tokens):      {s[long_key]:.1%}")
print(f"Mean PPL wm:             {s['mean_ppl_wm']:.2f}")
print(f"Mean PPL uwm:            {s['mean_ppl_uwm']:.2f}")
print(f"PPL ratio (wm/uwm):      {s['ppl_ratio_wm_over_uwm']:.3f}")

## Cell 4. Length curves (~5 min, no GPU work, post-hoc on existing corpus)

Output: `results/length_curves_llama.json`

In [ ]:
!python scripts/eval_length.py --model meta-llama/Llama-3.1-8B-Instruct

In [ ]:
import json
with open('results/length_curves_llama.json') as f:
    lc = json.load(f)
print(f"{'tokens':>8}  {'TPR':>6}")
for row in lc:
    bar = '#' * int(row['tpr'] * 25)
    print(f"{row['n_tokens']:>8}  {row['tpr']:>6.3f}  {bar}")

## Cell 5. Robustness sweep (~15 min, basic conditions only)

Word substitution at 5/10/15/20 %, plus 10 % token insertion and 10 % token deletion. **Skips the LLM paraphrase attack** (which would require reloading the 8B model and ~60 extra minutes).

Output: `results/robustness_llama.json`

In [ ]:
!python scripts/eval_robustness.py --model meta-llama/Llama-3.1-8B-Instruct

In [ ]:
import json
with open('results/robustness_llama.json') as f:
    rob = json.load(f)
print(f"{'Condition':<32} {'TPR':>6}")
for cond, vals in rob.items():
    bar = '#' * int(vals['tpr_at_1pct_fpr'] * 25)
    print(f"{cond:<32} {vals['tpr_at_1pct_fpr']:>6.1%}  {bar}")

## Cell 6. Download the result files

Zips just the JSON + npz files needed to fill in the report. Save the downloaded zip and either (a) drop it into `results/` on the local repo, or (b) commit and push so the report build picks them up.

If you re-open this notebook later, your results survive in `/content/llm-invisible-watermarking/results/` only as long as the Colab session lives, so download before disconnecting.

In [ ]:
import zipfile, os
from datetime import datetime
from pathlib import Path

timestamp = datetime.now().strftime('%Y%m%d_%H%M')
archive   = f'/content/results_llama_eval_{timestamp}.zip'

wanted = [
    'results/detection_llama_summary.json',
    'results/detection_llama_zscores.npz',
    'results/length_curves_llama.json',
    'results/robustness_llama.json',
    'results/robustness_llama_zscores.npz',
]

with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as zf:
    for fp in wanted:
        if Path(fp).exists():
            zf.write(fp)
            print(f'  added {fp}')
        else:
            print(f'  MISSING {fp}. make sure earlier cells finished cleanly')

size_mb = Path(archive).stat().st_size / 1e6
print(f'\nCreated {archive}  ({size_mb:.2f} MB)')

from google.colab import files
files.download(archive)